# 30 · 离线试一试"校准打分"：先算，再建

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → 17 找相似病人 ✅ → 18 疾病介绍（RAG） ✅ → 19 基因工具 ✅ → 20 大考 ✅ → 21 正式评测 ✅ → 22 做成服务 ✅ → 23 和别人比 ✅ → 24 和大模型比 ✅ → 25 和 DeepRare 比 ✅ → 26 读病历 ✅ → 27 交付 ✅ → 28 时间外验证 ✅ → 29 外部验证 ✅ → [30 离线试打分 ✅] → 31 加进 Orphanet 的病
```

> **这一章的代码在分支 `feat/calibrated-scoring` 里。运行这本笔记不花钱**：不调用任何大模型。系统的回答从 MLflow 里已经记下的考试读出来，证据用本机的工具重新算。全部跑完约 2 分钟。
>
> **先说结论：这条路没走通。** 这一章把它是怎么试的、为什么没走通、接下来怎么改，都照实写下来。

## 背景：输在"选"，那就换个"选"法？

第 29 章第 7 步发现：答错的题里，大多数时候**正确答案已经在名单里了，只是没排到第一**。比如 HMS 上，直接猜对、v1.0 错的 15 道题里，有 12 道正确答案就在 v1.0 自己的前 10 名里。

现在"从名单里选哪个当第一"，全靠大模型读完证据凭感觉定。路线图（[`docs/roadmap.md`](../roadmap.md) 第 5.2 节）设想的第三阶段是：**把"选"改成按证据打分，每项证据占多大分量，从过去的病例里学出来**。打个比方，就像银行算信用分：收入、负债、逾期次数各占多少分，是从几万个贷款人身上学出来的。

但这是个大改动：要重写 agent 的整个后半段。动手之前，先问一句：**这个办法到底能不能比现在的 agent 选得准？**

**公司里的做法**：这叫**离线评测**（offline evaluation），也叫"回放"（replay）。新的排序办法上线前，先拿历史记录算一遍：如果当时用的是新办法，结果会怎样。算下来不行，就不上线。搜索、推荐、广告团队都是这么做的。**打个比方**：新菜谱先在家试做一次，好吃再上菜单。

我们正好有条件这么做：每一次考试，每个系统对每道题的回答都记在 MLflow 里（第 08 章）。

## 这一章做的 4 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 把几路名单合成一个候选池，每个候选算一行证据 | 已记下的回答 + 本机的工具 | 打分的原料 |
| 2 | 从数据里学每项证据的分量 | scikit-learn 的逻辑回归 | 打分 |
| 3 | 换一个来源检验 | 轮流留出一个来源；同一来源五折当参照 | 学到的分量能不能用到没见过的病人身上 |
| 4 | 看结果、定下一步 | 逐题配对 | 这条路走不走 |

**花费**：0。

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
import logging, warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # 一条和我们无关的提示
logging.getLogger("mlflow").setLevel(logging.WARNING)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = r.stdout + r.stderr
    print("\n".join(line for line in out.splitlines() if "INFO mlflow" not in line))
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 候选池：几路名单合在一起，每个候选一行证据

### 在干什么
对每一道题，把三路名单合成一个**候选池**，再给池子里的每一个病算一行**证据**。

### 背景
三路名单是：

| 哪一路 | 是什么 | 从哪来 |
|---|---|---|
| 模型自己猜 | 不给任何工具，让大模型凭知识列 10 个病（就是"大模型直接猜"那个系统） | MLflow 里记下的回答，**不重新问模型** |
| 症状排序 | 按症状相似度给所有遗传病排名，取前 20（第 10 章的工具） | 本机现算 |
| 相似病人 | 找症状最像的已发表病人，取他们的诊断，前 10 个（第 17 章的工具） | 本机现算 |

题目来自开发用的四个来源（路线图第 5.1 节）：dev-200、HMS、RAMEDIS、MME，共 952 道，记在 [`evals/calibration.toml`](../../evals/calibration.toml)，里面写着每个来源读哪几次考试的记录。**留出集和考卷一道都不碰**：配置里要是写了它们，命令会直接拒绝。

### 用了什么技术，为什么

**两段式系统**：先用几个便宜、宽松的办法"捞"出一批候选（工业界叫**召回**，recall），再仔细排序（**精排**，ranking）。搜索引擎、购物推荐都是这个结构。

每个候选的那一行证据：

| 列 | 意思 | 打个比方 |
|---|---|---|
| model / phenotype / patients | 三路名单各把它排在第几（`-` = 没提名它） | 三位评委各给的名次 |
| explained | 病人的症状里，这个病能解释的比例 | 简历和岗位要求对上了几成 |
| contradicted | 资料里写着"这个病的病人查过、没有"，可这位病人有的症状 | 明确不符的条件 |
| missing | 这个病一半以上病人都有、这位病人却没有的症状个数 | 岗位最看重、他却没有的技能 |
| annotated | 知识库里这个病一共记了几条症状 | 这份简历写得有多详细 |
| prevalence | Orphanet 的患病率档：common（十万分之一以上）、uncommon、rarest（不到百万分之一）、unknown | 这个病常不常见 |

**和以前一样遮挡**：算证据时，遮住这道题自己的出处论文（第 21 章）。

### 公司里什么情况怎么做
- 精排之前，先看**召回够不够**：正确答案连池子都没进，排序再好也没用
- 每个候选的特征，要和线上系统看到的**完全一样**，不能偷看答案。这里的证据全部来自同一套工具、同样的遮挡规则

### 执行
先看一道真题：**HMS 第 3 个病人**，确诊是**白塞病**（一种反复口腔溃疡、眼睛发炎的免疫病），有 43 个症状。大模型直接猜对了这道，v1.0 和 v1.1 都错了：

In [ ]:
run("uv run raredx eval calibrate --show RareBench_HMS_003")

### 结果怎么看
- 这道题的池子里有 35 个候选，按"模型自己猜 → 症状排序 → 相似病人"的顺序列出来；最后一列 `yes` 标出正确答案
- **模型自己猜**：白塞病排**第 1**，后面是结节病、红斑狼疮、肉芽肿性多血管炎，都是相对常见的免疫病
- **症状排序**：前 20 名里**根本没有白塞病**。排第一的"家族性肾性低尿酸血症"在知识库里只记了 1 条症状
- 白塞病自己：知识库只记了 **15 条**症状，能解释病人 43 个症状里的 **5%**。患病率是 common
- `v1.1` 一列：v1.1 把白塞病排在**第 2**，第 1 是 **Yao 综合征**（能解释 20%，记了 20 条）

这就是第 29 章说的"选"的问题，摆成一行一行的证据看得更清楚：**正确答案在池子里，模型也提名了它，可是按"像不像"算，它比不过一个记得更全的罕见病。**

### 面试怎么说
> "For each case I pooled three routes — the model's own answer read back from a recorded run, the phenotype ranking and the nearest published patients — and gave every candidate one row of evidence computed with the same tools and the same paper mask the agent uses. In a typical loss the right answer is in the pool and even nominated first by the model, but explains only 5% of the findings because HPO annotates it thinly."

---

## 第 2 步 · 学分量：逻辑回归

### 在干什么
从开发用的 952 道题里学：**一行证据里，每一项该占多大分量**，才能让正确答案得分最高。

### 背景
每道题的池子里平均 36 个候选，只有一个是对的。把所有候选都标上"对 / 错"，就是一份学习材料：33,932 行。

### 用了什么技术，为什么

| 选择 | 为什么 |
|---|---|
| **逻辑回归**（scikit-learn） | 每项证据乘一个分量，加起来就是分数。分量**能直接读出来**："多解释一成症状加多少分""极罕见加多少分"。病例只有几百道，复杂的模型（神经网络、梯度提升树）学不稳，也说不清为什么 |
| 每项证据**放两次**：原值，和"它比同一道题里最好的候选差多少" | 选的是"这道题里哪个最好"，不是"这个病绝对得多少分"。**打个比方**：选班长看的是你比别的候选人多几票，不是你自己拿了多少票。工业界的排序模型（learning to rank）都看这种同组相对值 |
| **每个来源同样分量** | RAMEDIS 有 624 道，MME 只有 40 道。不这样做，RAMEDIS 一家说了算 |

**为什么用 scikit-learn**：Python 里最通用的机器学习库，公司里做这类表格数据的第一版模型几乎都用它。

### 公司里什么情况怎么做
- 排序模型的第一版，常用逻辑回归或评分卡：**好解释、好检查**，出了问题能看出是哪项证据在起作用
- 样本来源不均时，按来源加权，不让最大的那一份数据定规则

### 执行
学分量和检验在同一条命令里，下一步一起跑。

---

## 第 3 步 · 换一个来源检验：学到的分量搬得走吗

### 在干什么
检验学到的分量，在**没学过的那一类病人**上管不管用。

### 背景
这正是第 29 章暴露的问题：系统在一类病人上调好了，换一类病人就不行。开发用的四个来源，病人很不一样：

| 来源 | 是什么病人 | 谁更强（第 29 章） |
|---|---|---|
| dev-200 | 发表过的单基因病 | 工具 |
| HMS | 门诊病人，很多免疫病 | 模型自己 |
| RAMEDIS | 代谢病数据库，四分之一是苯丙酮尿症 | 模型自己 |
| MME | 极罕见病，模型几乎不认识 | 工具 |

### 用了什么技术，为什么

| 检验 | 怎么做 | 打个比方 | 说明什么 |
|---|---|---|---|
| **换来源**（主要的） | 用三个来源学分量，在第四个上打分；四个来源轮一遍 | 用三位老师出的题练，第四位老师出的题考 | 能不能用到**没见过的那一类病人**身上 |
| **同来源**（参照） | 同一个来源的题分成五份，四份学、一份打分，轮一遍 | 同一位老师出的题，分五份，四份练一份考 | **偏乐观**：同一位老师爱考的病会反复出现。只说明"这些证据最多能做到多好" |

另外每种检验都试两个版本：**只用证据**；**证据 + v1.1 的排名**（把 v1.1 怎么排的也当成一项证据）。

### 公司里什么情况怎么做
- 这叫**分组交叉验证**（grouped cross-validation），这里一组就是一个来源。医疗 AI 里叫**留一中心验证**（leave-one-site-out）：在几家医院的数据上学，在另一家医院上验，审评最看重这一项
- 随机打散再切，会让同一来源的题同时出现在学和考里，分数虚高。**切法要和"上线以后会遇到什么"一致**：上线以后遇到的，是没见过的医院、没见过的病人

### 执行
跑完整的一遍，约 2 分钟，不花钱。逐题结果写进 `docs/calibration_hits.tsv`：

In [ ]:
run("uv run raredx eval calibrate --out docs/calibration_hits.tsv")

### 结果怎么看
第一张表是每个来源的第 1 名命中率：

- `in pool`：正确答案进没进池子（召回）。dev-200 67%、HMS 60%、RAMEDIS 85%、MME 88%：大多数题，正确答案就在池子里
- `model`、`v1.0`、`v1.1`：三个已有系统，从记录里读的
- `scorer`：只用证据、**换来源**学的分量
- `scorer+v1.1`：证据加上 v1.1 的排名、**换来源**
- `/same`：**同来源**的参照

第二张是逐题配对。最后一行是学到的分量，从大到小。下一步细看。

---

## 第 4 步 · 结果：没过

### 在干什么
回答这一章的问题：**校准打分能不能比 v1.1 选得准？** 路线图第 9 节事先写过：如果连用已有证据校准打分都胜不过现在的 agent，打分这条路就要改。

### 执行

In [ ]:
import csv
from raredx.evals.scoring import p_value, sign_test

ROWS = list(csv.DictReader(open(ROOT / "docs/calibration_hits.tsv", encoding="utf-8"), delimiter="\t"))
SOURCES = ("dev-200", "rarebench-hms", "rarebench-ramedis", "rarebench-mme")
COLUMNS = {"v1.1": "v1.1", "scorer": "只用证据，换来源", "scorer/same": "只用证据，同来源",
           "scorer+v1.1": "加 v1.1，换来源", "scorer+v1.1/same": "加 v1.1，同来源"}

def share(rows, column):
    return sum(r[column] == "1" for r in rows) / len(rows)

print(f"{'第 1 名':<18}" + "".join(f"{s.replace('rarebench-', ''):>10}" for s in SOURCES))
for column, name in COLUMNS.items():
    print(f"{name:<16}" + "".join(f"{share([r for r in ROWS if r['source'] == s], column + '@1'):>10.1%}" for s in SOURCES))

print("\n逐题配对，第 1 名（只有前者对 : 只有 v1.1 对）")
for column in ("scorer", "scorer+v1.1", "scorer+v1.1/same"):
    cells = []
    for s in SOURCES:
        mine = [r for r in ROWS if r["source"] == s]
        w = sum(r[column + "@1"] == "1" and r["v1.1@1"] == "0" for r in mine)
        l = sum(r["v1.1@1"] == "1" and r[column + "@1"] == "0" for r in mine)
        cells.append(f"{w}:{l} {p_value(sign_test(w, l))}")
    print(f"  {COLUMNS[column]:<14}" + "   ".join(f"{s.replace('rarebench-', '')} {c}" for s, c in zip(SOURCES, cells)))

### 结果怎么看
**1. 换个来源就失灵。** 只用证据，同来源能到 35%（dev-200）、45%（RAMEDIS）、73%（MME）；换个来源就掉到 20% 多（MME 掉到 30%）。dev-200、RAMEDIS、MME 上**显著输给 v1.1**；HMS 上 5 : 13（p = 0.096），也是输，只是没到显著。

**2. 就算最好的情况，只用证据也不如 v1.1。** 同来源的参照，dev-200 35% 对 v1.1 的 41.5%，HMS 29.5% 对 37.5%。RAMEDIS 同来源 45% 看着高，那是因为四分之一是苯丙酮尿症，同一个病在五份里反复出现，学的是"这份题爱考什么"。**大模型的判断里有我们这行证据没有的东西。**

**3. 加上 v1.1 的排名。** 换来源：HMS、RAMEDIS 上和 v1.1 打平，MME 略低，dev-200 上**显著更差**（4 : 21）。同来源：四个来源方向上都比 v1.1 高（43.0% 对 41.5%、42.0% 对 37.5%、40.2% 对 30.6%、77.5% 对 72.5%），但逐题配对**只有 RAMEDIS 显著**（99 : 39），而 RAMEDIS 的同来源数字本来就偏乐观。所以只能说：**有同一类病人的、标好答案的病例来校准，在模型判断之上再加一层打分可能有用，值得再验证；换一类病人，这层打分没用，甚至有害。**

**为什么搬不走：看学到的分量。** 上一步输出的最后一行，分量最大的几项是：比同题最好的多解释症状 +0.93、缺少的常见症状 −0.71、**极罕见 +0.68**、**记得越全越好 +0.62**。在 dev-200 和 MME 上这是对的：答案多是记得很全的极罕见单基因病。在 HMS 和 RAMEDIS 上正好相反：白塞病这种相对常见、记得很少的病才是答案。**几个来源要的东西方向相反，学出一套分量，必然顾此失彼。** 这和 v1.0 的毛病是同一个毛病，只是换成了一个用数据学出来的版本。

**结论：没过。** 不把"学出来的打分"做进 agent 来代替模型的判断。

### 面试怎么说
> "Before building a learned re-ranker into the agent I tested it offline on recorded runs, validated by leaving one data source out. Trained on the same source it reached 35–45%; on an unseen source it fell to 22–30%, below the LLM agent on all four. The weights it learned — prefer richly annotated, ultra-rare diseases — are right for published single-gene cases and wrong for clinic patients. So I didn't build it; the experiment cost nothing and saved a rewrite."

---

## 第 5 步 · 接下来怎么改

### 这一章改变了什么
路线图第 5.2 节的第三阶段（"用数据校准的打分"）**不再代替模型的判断**：

| 原来的设想 | 改成 |
|---|---|
| 学一套分量，按分数选第一 | **模型复核做主**（v1.1 现在就是这样） |
| — | 学出来的分数只做两件事：① 估计**把握程度**（哪些答案可以先信）；② 有同一来源的少量标注病例时，**试一试按来源校准**（第 3 点结果：同来源时方向上都比模型高，但只有 RAMEDIS 显著，要再验证） |

第 ② 条，正好对上新外部考卷的设计（路线图第 5.3 节）：考卷切出来的 **60 道调试题**，就是"同一来源的少量标注病例"。liteOdyssey 在真实病人上也是这么做的：先拿 50 个病人调，再在剩下 465 个上考。

### 不受影响的
- **第 31 章：加进 Orphanet 的病**。这一章再次看到：白塞病在 OMIM 那边只记了 15 条症状，Orphanet 那边记了 85 条。这是补知识，和"怎么选"无关，哪个来源都用得上
- 第 33 章：病人资料补全（年龄、起病、家族史）

### 公司里什么情况怎么做
- **离线实验没过，就不上线，并把结论写下来**。下一次有人想到同一个点子，可以直接查到"试过，为什么不行"
- 一个办法在某类数据上不行，先看清楚是"根本不行"还是"换了地方不行"。这里是后者，所以留下了"按来源校准"这条路，但它还要在新数据上验证

---

## 本章小结

| 步骤 | 用的技术 | 在公司里的作用 |
|---|---|---|
| 1 候选池和证据行 | 两段式（召回 + 精排）；已记录的回答 + 本机工具，同样的遮挡 | 先看召回够不够，特征和线上一致 |
| 2 学分量 | scikit-learn 逻辑回归；同题相对值；每个来源同样分量 | 好解释的第一版排序模型 |
| 3 换来源检验 | 分组交叉验证（留一个来源）；同来源五折当参照 | 检验能不能用到没见过的病人身上 |
| 4 结果 | 逐题配对 | 离线没过，就不上线 |
| 5 接下来 | 改路线图：模型做主，分数用于把握程度和按来源校准 | 把"为什么不做"也记下来 |

**一句话**：动手重写 agent 之前，先用记下的考试离线试了"学出来的打分"：同一类病人上它可能有用，换一类病人就输给现在的 agent，因为几类病人要的东西方向相反。所以不拿它代替模型的判断，只留作把握程度和按来源校准。

## 面试题

> **Q: Why test the re-ranker offline instead of just building it?**
>
> A: "Because every answer every system gave was already recorded in MLflow, a replay cost nothing, while building it meant rewriting the back half of the agent. The replay said no: below the agent on every unseen source. That is the same discipline as offline evaluation before an online test in search or recommendation."

> **Q: Why leave one source out rather than a random split?**
>
> A: "A random split puts the same source's patients — and the same diseases — on both sides, so it measures how well you fit that source. In deployment the system meets a hospital it has never seen. Leaving one source out is the honest version of that, the same idea as leave-one-site-out validation in clinical AI."

## 下一章

**31 · 加进 Orphanet 的病**：知识库现在只读了 OMIM 的那一部分。用 Mondo 疾病本体把 OMIM 和 Orphanet 的编号对齐，把 Orphanet 的 4,355 种病和它们更全的症状记录加进来，并定好防泄露的规则。